In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('https://raw.githubusercontent.com/kimurasaika/car_price_prediction/refs/heads/main/data2.csv')
# df = pd.read_excel('/content/dataaaaaaaa.xlsx', sheet_name='Sheet1')
# df = df.drop(columns=['obs','month','month_num','year','EXI_IDX','PPI_IDX','IMI_IDX','policy','OIL_IDX','publicdebt','govdebt','SUGAR_IDX'])
# df['IMI_IDX'] = np.log(df['IMI_IDX'])
df = df.drop(columns=['obs','month','month_num','year','PPI_IDX'])

df.head()

,CPI_IDX,EX_AVG_USD,EXI_IDX,IMI_IDX,M2,policy,oilwtiprice,unemployment,GOLD
0,90.16,32.7344,94.8,87.0,"16,947,602",2.00,48.24,1.06,19407.69
1,90.26,32.5734,95.0,87.2,"17,113,914",2.00,49.76,0.82,19002.08
2,90.42,32.6314,95.1,86.7,"17,193,104",1.75,47.60,0.99,18278.00
3,90.43,32.5095,95.2,87.3,"17,166,475",1.50,59.63,0.85,18460.53
4,90.58,33.5519,95.5,88.0,"17,190,904",1.50,60.30,0.93,19052.78


In [ ]:
df.isna().sum()

,0
CPI_IDX,0
EX_AVG_USD,0
EXI_IDX,0
IMI_IDX,0
M2,0
policy,0
oilwtiprice,0
unemployment,15
GOLD,0


In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
imputer = IterativeImputer(max_iter = 10 , random_state=0)
df_imputed = imputer.fit_transform(df)
df_imputed = pd.DataFrame(df_imputed, columns=df.columns)
df_imputed.head()

ValueError: could not convert string to float: '16,947,602'

In [ ]:
df_imputed.isna().sum()

In [ ]:
df_imputed['unemployment']

In [ ]:
print(df['unemployment'].mean())
print(df_imputed['unemployment'].mean())

In [ ]:
df_imputed['unemployment'] = np.log1p(df_imputed['unemployment'])
# df_imputed['GOLD'] = np.log1p(df_imputed['GOLD'])
df_imputed.head()


In [ ]:
df_imputed['CPI_IDX'] = np.log1p(df_imputed['CPI_IDX'])


In [ ]:
col = df_imputed.columns
for i in col:
    sns.histplot(x= i,data = df_imputed,color = "red",kde=True)
    plt.xticks(rotation = 45)
    plt.title("distribution of "+i)
    plt.show()

In [ ]:
nu_df = df_imputed.select_dtypes(include = [np.number])
plt.figure(figsize = (12,9))
corr = nu_df.corr(method = 'spearman')
sns.heatmap(corr , fmt = '.2f' , annot = True , cmap = 'coolwarm' )
plt.title('Correlation Heatmap Pearson')

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import matplotlib.pyplot as plt
import pylab
from scipy import stats


def calculate_vif(df):
    df_with_const = sm.add_constant(df)
    vif_df = pd.DataFrame()
    vif_df['Column'] = df.columns
    vif_df['VIF'] = [variance_inflation_factor(df_with_const.values, i + 1) for i in range(df.shape[1])]
    return vif_df
# Backward elimination function
def backward_elimination(X, y, threshold_out=0.05, verbose=True):
    included = list(X.columns)
    while included:
        model = sm.OLS(y, sm.add_constant(X[included])).fit()
        pvalues = model.pvalues.iloc[1:]  # Exclude constant
        worst_pval = pvalues.max()
        if worst_pval > threshold_out:
            worst_feature = pvalues.idxmax()
            included.remove(worst_feature)
            if verbose:
                print(f'Remove {worst_feature} with p-value {worst_pval:.6f}')
        else:
            break
    return included

def stepwise_selection(X, y, threshold_in=0.05, threshold_out=0.05, verbose=True):

    included = []  # เริ่มจากโมเดลว่าง
    excluded = list(X.columns)  # ตัวแปรทั้งหมดที่ยังไม่ได้เลือก

    while True:
        changed = False

        # Forward Selection: ลองเพิ่มตัวแปรที่เหลือทีละตัว
        new_pval = pd.Series(index=excluded, dtype=float)
        for new_feature in excluded:
            model = sm.OLS(y, sm.add_constant(X[included + [new_feature]])).fit()
            new_pval[new_feature] = model.pvalues[new_feature]

        best_pval = new_pval.min()
        if best_pval < threshold_in:  # ถ้ามีตัวแปรที่ p-value < threshold_in
            best_feature = new_pval.idxmin()
            included.append(best_feature)
            excluded.remove(best_feature)
            changed = True
            if verbose:
                print(f'Add {best_feature} with p-value {best_pval:.6f}')

        # Backward Elimination: ตรวจสอบตัวแปรที่อยู่ในโมเดล
        if included:
            model = sm.OLS(y, sm.add_constant(X[included])).fit()
            pvalues = model.pvalues.iloc[1:]  # Exclude constant
            worst_pval = pvalues.max()
            if worst_pval > threshold_out:  # ถ้ามีตัวแปรที่ p-value > threshold_out
                worst_feature = pvalues.idxmax()
                included.remove(worst_feature)
                excluded.append(worst_feature)
                changed = True
                if verbose:
                    print(f'Remove {worst_feature} with p-value {worst_pval:.6f}')

        # ถ้าไม่มีอะไรเปลี่ยนแปลง (ไม่เพิ่มหรือลบตัวแปร) ออกจาก loop
        if not changed:
            break

    return included


def diagnostic_plots(df, variable):
    # function to plot a histogram and a Q-Q plot
    # side by side, for a certain variable

    plt.figure(figsize=(15,6))
    plt.subplot(1, 2, 1)
    df[variable].hist()

    plt.subplot(1, 2, 2)
    stats.probplot(df[variable], dist="norm", plot=pylab)

    plt.show()

In [ ]:
X = X = df_imputed.drop(columns=['CPI_IDX'])
y = df_imputed['CPI_IDX']

In [ ]:
selected_features = backward_elimination(X, y)
# selected_features = stepwise_selection(X, y)
print("\nSelected features (after backward elimination):", selected_features)
X_selected = X[selected_features]

In [ ]:
X_scale = X_selected.copy()
# y = np.log(y)
# X_scale.drop(columns=['FOOD_IDX','M2','LAST_XRP'], inplace=True)
for i in X_scale.columns:
    X_scale[i] = (X_scale[i] - X_scale[i].min()) / (X_scale[i].max() - X_scale[i].min())
X_const = sm.add_constant(X_scale)
model = sm.OLS(y, X_const).fit()
print(model.summary())
residuals = model.resid

y_pred = model.predict(X_const)
plt.figure(figsize=(10, 6))
plt.scatter(y_pred, y, alpha=0.5)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
plt.xlabel('Predicted CPI')
plt.ylabel('Actual CPI')
plt.title('Actual vs Predicted (Linearity Check)')
plt.show()

from statsmodels.stats.stattools import durbin_watson
dw_stat = durbin_watson(model.resid)
print(f'Durbin-Watson statistic: {dw_stat:.3f}')

from statsmodels.stats.diagnostic import het_breuschpagan
bp_test = het_breuschpagan(model.resid, X_const)
labels = ['LM Statistic', 'LM-Test p-value', 'F-Statistic', 'F-Test p-value']
print(dict(zip(labels, bp_test)))

vif_result = calculate_vif(X_const.drop(columns=['const']))
print("\nVIF Results:")
print(vif_result)

shapiro_test = stats.shapiro(residuals)
print(f'Shapiro-Wilk Test: Statistic={shapiro_test.statistic:.3f}, p-value={shapiro_test.pvalue:}')

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan
import statsmodels.api as sm
from sklearn.preprocessing import LabelEncoder , OneHotEncoder
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


X = X = df_imputed.drop(columns=['CPI_IDX'])
y = df_imputed['CPI_IDX']
# selected_features = stepwise_selection(X, y)
# selected_features = backward_elimination(X, y)
# X = X[selected_features]

le = LabelEncoder()
X['policy'] = le.fit_transform(X['policy'])
policy_dummies = pd.get_dummies(X['policy'], prefix='policy', drop_first=True)

X = pd.concat([X.drop(columns=['policy']), policy_dummies], axis=1)


In [ ]:
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import RobustScaler , StandardScaler , MinMaxScaler , MaxAbsScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
import pandas as pd

scaler = RobustScaler()

model = LinearRegression()

loo = LeaveOneOut()

y_true = []
y_pred = []
# LOOCV
for train_index, test_index in loo.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y[train_index], y[test_index]
    # print(f"train_index: {train_index}, test_index: {test_index}")
    # print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")
    # print(f"y_train shape: {y_train.shape}, y_test shape: {y_test.shape}")

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model.fit(X_train_scaled, y_train)

    y_pred_i = model.predict(X_test_scaled)

    y_true.append(y_test.item() if isinstance(y_test, np.ndarray) and y_test.size == 1 else y_test)
    y_pred.append(y_pred_i.item() if isinstance(y_pred_i, np.ndarray) and y_pred_i.size == 1 else y_pred_i)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

r2ln = r2_score(y_true, y_pred)
mseln = mean_squared_error(y_true, y_pred)
maeln = mean_absolute_error(y_true, y_pred)
rmseln = np.sqrt(mseln)
print(f'R-squared Score (LOOCV): {r2ln:.3f}')
print(f'Mean Squared Error (LOOCV): {mseln:.3f}')
print(f'Mean Absolute Error (LOOCV): {maeln:.3f}')
print(f'Root Mean Squared Error (LOOCV): {rmseln:.3f}')


In [ ]:

X = X.drop(columns=['EXI_IDX','IMI_IDX'])
model = LinearRegression().fit(X, y)
residuals = y - model.predict(X)
y_predln = model.predict(X)
# test
dw_stat = durbin_watson(residuals)
print(f'Durbin-Watson statistic: {dw_stat}')
shapiro_res = stats.shapiro(residuals)
print(f's normality: p-value={shapiro_res.pvalue}')
X_const = sm.add_constant(X)
bp_test = het_breuschpagan(residuals, X_const)
labels = ['Lagrange multiplier statistic', 'p-value', 'f-value', 'f p-value']
print(dict(zip(labels, bp_test)))

X_const_numeric = X_const.copy()
for col in X_const_numeric.columns:
    if X_const_numeric[col].dtype == 'bool':
        X_const_numeric[col] = X_const_numeric[col].astype(int)

vif_result = calculate_vif(X_const_numeric.drop(columns=['const']))
print("\nVIF Results:")
print(vif_result)

r2ln = r2_score(y,y_predln)
mseln = mean_squared_error(y, y_predln)
maeln = mean_absolute_error(y, y_predln)
rmseln = np.sqrt(mseln)

print(f'R-squared Score (LN): {r2ln:.3f}')
print(f'Mean Squared Error (LN): {mseln:.3f}')
print(f'Mean Absolute Error (LN): {maeln:.3f}')
print(f'Root Mean Squared Error (LN): {rmseln:.3f}')

In [ ]:
poly = PolynomialFeatures(degree=2 , include_bias=False)
X_poly = poly.fit_transform(X)

X_numeric = X.copy()
for col in X_numeric.columns:
    if X_numeric[col].dtype == 'bool':
        X_numeric[col] = X_numeric[col].astype(int)

X_poly_const = sm.add_constant(pd.DataFrame(X_poly, columns=poly.get_feature_names_out(X_numeric.columns)))

residuals = y - LinearRegression().fit(X_poly, y).predict(X_poly)
# test
dw_stat = durbin_watson(residuals)
print(f'Durbin-Watson statistic: {dw_stat}')
shapiro_res = stats.shapiro(residuals)
print(f'Residuals normality: p-value={shapiro_res.pvalue}')
bp_test = het_breuschpagan(residuals, X_poly_const)
labels = ['Lagrange multiplier statistic', 'p-value', 'f-value', 'f p-value']
print(dict(zip(labels, bp_test)))
X_poly_const_numeric = X_poly_const.copy()
for col in X_poly_const_numeric.columns:
    if X_poly_const_numeric[col].dtype == 'bool':
        X_poly_const_numeric[col] = X_poly_const_numeric[col].astype(int)

vif_result = calculate_vif(X_poly_const_numeric.drop(columns=['const']))
print("\nVIF Results:")
print(vif_result)

In [ ]:
print(X_poly.shape)
print(X.shape)

In [ ]:
    import pandas as pd
    from sklearn.linear_model import LinearRegression
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np

    # Sample data
    np.random.seed(42)
    X = np.random.rand(100, 1) * 10
    y_true = 2 * X + 1 + np.random.randn(100, 1) * (X / 2) # Introducing heteroscedasticity
    y_true_homo = 2 * X + 1 + np.random.randn(100, 1) * 2 # Homoscedastic example

    # Fit a linear regression model
    model = LinearRegression()
    model.fit(X, y_true)
    y_pred = model.predict(X)

    # Calculate residuals
    residuals = y_true - y_pred

In [ ]:
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x=y_pred.flatten(), y=residuals.flatten())
    plt.axhline(y=0, color='r', linestyle='--')
    plt.title('Residuals vs. Predicted Values (Funnel Test)')
    plt.xlabel('Predicted Values')
    plt.ylabel('Residuals')
    plt.grid(True)
    plt.show()